In [ ]:
import sys
import os

# Add parent directory to path so we can import DatasetLoader
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
from DatasetLoader import cub_v2 as cub
from DatasetLoader import CXR as cxr
import NetworkManager
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report


In [ ]:
# --------------------- EDIT THIS TO CHANGE DATASET --------------------- #
DATASET = "cub"  # Options: "cxr" or "cub"
BASE_WEIGHTS_DIR = "./drive_folder/Bridging Human and Model Attention_ Explainability Analysis of CNN, Mamba, and ViT Architectures with Gaze-Based Validation"


In [ ]:
DEFAULT_BATCH_SIZE   = 1
DEFAULT_IMG_SIZE     = 448
#dummy values since we are not training
DEFAULT_BASE_LR      = 0.001
DEFAULT_EPOCHS       = 95
DEFAULT_MOMENTUM     = 0.9
DEFAULT_WEIGHT_DECAY = 1e-4
DEFAULT_GPU_ID       = 0





dataset_folder = "CXR_weights" if DATASET == "cxr" else "CUB_weights"
net_options_cnn = {
    'net_choice': "ResNet",
    'model_choice': 50,
    'epochs': DEFAULT_EPOCHS,
    'batch_size': DEFAULT_BATCH_SIZE,
    'base_lr': DEFAULT_BASE_LR,
    'weight_decay': DEFAULT_WEIGHT_DECAY,
    'momentum': DEFAULT_MOMENTUM,
    'img_size': DEFAULT_IMG_SIZE,
    'device': torch.device('cuda:'+str(DEFAULT_GPU_ID) if torch.cuda.is_available() else 'cpu'),
    'model_type': 50,
    'checkpoint_path': f'{BASE_WEIGHTS_DIR}/CNN/{dataset_folder}/ResNet50.pkl',
}

net_options_vit_base = {
    'net_choice': "Transformer",
    'model_choice': 'vit_base_patch16_224',
    'epochs': DEFAULT_EPOCHS,
    'batch_size': DEFAULT_BATCH_SIZE,
    'base_lr': DEFAULT_BASE_LR,
    'weight_decay': DEFAULT_WEIGHT_DECAY,
    'momentum': DEFAULT_MOMENTUM,
    'img_size': DEFAULT_IMG_SIZE,
    'device': torch.device('cuda:'+str(DEFAULT_GPU_ID) if torch.cuda.is_available() else 'cpu'),
    'model_type': 'vit_base_patch16_224',
    'save_folder_path': './model_save',
    'checkpoint_path': f'{BASE_WEIGHTS_DIR}/Transformer/{dataset_folder}/vit_base_patch16_224_Unfrozen.pkl',
}

net_options_vit_base_F = {
    'net_choice': "Transformer",
    'model_choice': 'vit_base_patch16_224',
    'epochs': DEFAULT_EPOCHS,
    'batch_size': DEFAULT_BATCH_SIZE,
    'base_lr': DEFAULT_BASE_LR,
    'weight_decay': DEFAULT_WEIGHT_DECAY,
    'momentum': DEFAULT_MOMENTUM,
    'img_size': DEFAULT_IMG_SIZE,
    'device': torch.device('cuda:'+str(DEFAULT_GPU_ID) if torch.cuda.is_available() else 'cpu'),
    'model_type': 'vit_base_patch16_224',
    'save_folder_path': './model_save',
    'checkpoint_path': f'{BASE_WEIGHTS_DIR}/Transformer/{dataset_folder}/vit_base_patch16_224_Frozen.pkl',
}

net_options_mamba = {
    'net_choice': "Mamba",
    'model_choice': "vim_base_patch16_224",
    'epochs': DEFAULT_EPOCHS,
    'batch_size': DEFAULT_BATCH_SIZE,
    'base_lr': DEFAULT_BASE_LR,
    'weight_decay': DEFAULT_WEIGHT_DECAY,
    'momentum': DEFAULT_MOMENTUM,
    'img_size': DEFAULT_IMG_SIZE,
    'device': torch.device('cuda:'+str(DEFAULT_GPU_ID) if torch.cuda.is_available() else 'cpu'),
    'checkpoint_path': f'{BASE_WEIGHTS_DIR}/Mamba/{dataset_folder}/Mambavim_base_patch16_224.pkl',
    'freeze_params': True,
    'model_type': "vim_base_patch16_224",
    'save_folder_path': './model_save'
}

cxr_dataset_options = cxr.dataset_options
cub_dataset_options = cub.dataset_options

cub_dataset_options['data_root'] = BASE_WEIGHTS_DIR + '/CUB/DATASET/'
cxr_dataset_options['data_root'] = BASE_WEIGHTS_DIR + '/CXR/'


In [ ]:
if DATASET == "cxr":
    train_loader, test_loader = cxr.get_dataloaders(batchsize=DEFAULT_BATCH_SIZE, data_dir=cxr_dataset_options['data_root'])
    dataset_options = cxr_dataset_options
elif DATASET == "cub":
    train_loader, test_loader = cub.get_dataloaders(batch_size=DEFAULT_BATCH_SIZE, root=cub_dataset_options['data_root'])
    dataset_options = cub_dataset_options

manager_cnn = NetworkManager.NetworkManager(net_options_cnn, dataset_options, train_loader, test_loader, checkpoint_path=net_options_cnn['checkpoint_path'], mode="eval")
manager_mamba = NetworkManager.NetworkManager(net_options_mamba, dataset_options, train_loader, test_loader, mode="eval")
#manager_vit_base = NetworkManager.NetworkManager(net_options_vit_base, dataset_options, train_loader, test_loader, mode="eval", checkpoint_path=net_options_vit_base['checkpoint_path'])
#manager_vit_base_F = NetworkManager.NetworkManager(net_options_vit_base_F, dataset_options, train_loader, test_loader, mode="eval", checkpoint_path=net_options_vit_base_F['checkpoint_path'])

In [ ]:
model_1 = manager_cnn.net
model_2 = manager_mamba.net
test_loader = manager_cnn.test_loader
device = manager_cnn.device

# Test the enseble
print('\n--- Detailed Evaluation ---')
#print(f'Class number: {n_class}')
with torch.no_grad():

    all_targets = []
    all_preds_scores = []
    all_preds_labels = []
    
    for imgs, labels, _ in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        logits_1 = model_1(imgs)
        logits_2 = model_2(imgs)

        # 1. Calcolo Punteggi per AUC (Probabilità Softmax)
        scores_1 = torch.softmax(logits_1, dim=1)
        scores_2 = torch.softmax(logits_2, dim=1)
        ensemble_scores = scores_1 + scores_2
        ensemble_scores = torch.softmax(ensemble_scores, dim=1)

        # 2. Etichette Previste (Predizioni)
        _, predicted_labels = torch.max(ensemble_scores, 1)
        # Accumula risultati su CPU
        all_targets.extend(labels.cpu().numpy())
        all_preds_scores.extend(ensemble_scores.cpu().numpy())
        all_preds_labels.extend(predicted_labels.cpu().numpy())
# Conversione in array NumPy
y_true = np.array(all_targets)
y_pred = np.array(all_preds_labels)
y_score = np.array(all_preds_scores)
# ==========================================================================
# METRICS
# ==========================================================================
        
# Accuracy
accuracy = accuracy_score(y_true, y_pred)
# Precision, Recall, F1-Score
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
# AUC (Area Under the ROC Curve)
try:
    # Multi-classe OVR (One-vs-Rest)
    auc = roc_auc_score(y_true, y_score, multi_class='ovr')
except ValueError as e:
    print(f"Attenzione: Impossibile calcolare l'AUC. {e}")
    auc = np.nan
print("--- Performance results ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (Macro Avg): {precision_macro:.4f}")
print(f"Recall (Macro Avg): {recall_macro:.4f}")
print(f"F1-Score (Macro Avg): {f1_macro:.4f}")
print(f"AUC (Multi-class OVR): {auc:.4f}")